# RAG: разделяем ошибки поиска и ответа

ML Mentor · starter notebook · решения в репозитории отсутствуют.


## Задача

Соберите небольшое воспроизводимое базовое RAG-решение и научитесь отвечать на главный диагностический вопрос: система не нашла нужный документ или нашла, но неправильно извлекла ответ?

Это намеренно простое локальное решение на TF-IDF и лексическом извлечении. Оно не имитирует качество LLM и нужно для проверки контура оценки до подключения дорогой генерации.

**Что должно получиться:** Recall@1/3 и MRR поиска, качество ответа и отказа от ответа, порог отказа, выбранный только по проверочной выборке, и взаимоисключающая таблица исходов: `retrieval_miss`, `bad_context`, `unsupported_claim`, `answering_failure`, `correct_abstention`, `success`.

Используется фиксированная адаптированная подвыборка SQuAD 2.0, лицензия CC BY-SA 4.0. Атрибуция, описание изменений, правило отбора и контрольная сумма находятся в `data/squad2/` и `data/datasets.json`. CPU достаточно; API-ключи не нужны. В локальной копии репозитория данные читаются без сети. Если открыть только notebook в Colab, подготовительная ячейка скачает официальный `dev-v2.0.json`, проверит SHA-256, воспроизведёт ту же подвыборку и проверит её итоговую SHA-256.

In [ ]:
import hashlib
import json
import re
import string
import urllib.request
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel


SQUAD_SOURCE_URL = "https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json"
SQUAD_SOURCE_SHA256 = "80a5225e94905956a6446d296ca1093975c4d3b3260f1d6c8f68bc2ab77182d8"
SQUAD_SUBSET_SHA256 = "053396eeff7f1c07d473f2bd62c4d0a28f5d34d37f49d5ffd7922a58fbb5494f"


def sha256(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def normalized_json(data: object) -> bytes:
    return (json.dumps(data, ensure_ascii=False, indent=2, sort_keys=True) + "\n").encode("utf-8")


def build_squad_subset(source: bytes) -> bytes:
    raw = json.loads(source)
    documents_payload = []
    impossible_examples = 0
    for article in raw["data"]:
        for paragraph in article["paragraphs"]:
            answerable = [qa for qa in paragraph["qas"] if not qa["is_impossible"]]
            impossible = [qa for qa in paragraph["qas"] if qa["is_impossible"]]
            if not answerable:
                continue
            selected = [answerable[0]]
            if impossible and impossible_examples < 50:
                selected.append(impossible[0])
                impossible_examples += 1
            qas = []
            for qa in selected:
                seen = set()
                answers = []
                for answer in qa["answers"]:
                    key = (answer["text"], answer["answer_start"])
                    if key not in seen:
                        seen.add(key)
                        answers.append({"text": key[0], "answer_start": key[1]})
                qas.append(
                    {
                        "answers": answers,
                        "is_impossible": qa["is_impossible"],
                        "question": qa["question"],
                        "source_qa_id": qa["id"],
                    }
                )
            documents_payload.append(
                {
                    "context": paragraph["context"],
                    "document_id": f"squad2-{len(documents_payload):03d}",
                    "qas": qas,
                    "title": article["title"],
                }
            )
            if len(documents_payload) == 100:
                break
        if len(documents_payload) == 100:
            break
    assert len(documents_payload) == 100 and impossible_examples == 50
    return normalized_json(
        {
            "attribution": "Rajpurkar, Jia et al. Stanford Question Answering Dataset 2.0.",
            "dataset": "SQuAD 2.0",
            "documents": documents_payload,
            "license": "CC-BY-SA-4.0",
            "modificationNotice": (
                "Deterministic educational subset: first 100 eligible contexts in source order; "
                "one answerable QA per context; one impossible QA for the first 50 eligible contexts. "
                "Duplicate answer spans removed and fields normalized; source text unchanged."
            ),
            "selection": {
                "answerable_per_document": 1,
                "documents": 100,
                "impossible_examples": 50,
                "policy": "source order; first answerable QA per document; first impossible QA for first 50 eligible documents",
            },
            "source_sha256": SQUAD_SOURCE_SHA256,
            "source_version": raw["version"],
        }
    )


def resolve_squad_subset() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        path = candidate / "data" / "squad2" / "squad2-rag-subset.json"
        if path.is_file() and sha256(path.read_bytes()) == SQUAD_SUBSET_SHA256:
            return path
    cache_path = Path("/tmp/ml-mentor-labs-datasets-v1/squad2/squad2-rag-subset.json")
    if cache_path.is_file() and sha256(cache_path.read_bytes()) == SQUAD_SUBSET_SHA256:
        return cache_path
    request = urllib.request.Request(SQUAD_SOURCE_URL, headers={"User-Agent": "ML-Mentor-Labs/1.0"})
    with urllib.request.urlopen(request, timeout=120) as response:
        assert urlparse(response.geturl()).hostname == "rajpurkar.github.io"
        source = response.read()
    assert sha256(source) == SQUAD_SOURCE_SHA256, "SQuAD source checksum changed"
    subset = build_squad_subset(source)
    assert sha256(subset) == SQUAD_SUBSET_SHA256, "SQuAD subset recipe changed"
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_bytes(subset)
    return cache_path


def normalize_answer(text: str) -> str:
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def token_f1(prediction: str, references: list[str]) -> float:
    pred_tokens = normalize_answer(prediction).split()
    if not references:
        return float(not pred_tokens)
    scores = []
    for reference in references:
        ref_tokens = normalize_answer(reference).split()
        common = sum((min(pred_tokens.count(token), ref_tokens.count(token)) for token in set(pred_tokens)))
        if not pred_tokens or not ref_tokens:
            scores.append(float(pred_tokens == ref_tokens))
            continue
        precision = common / len(pred_tokens)
        recall = common / len(ref_tokens)
        scores.append(0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall))
    return max(scores, default=0.0)


SQUAD_SUBSET_PATH = resolve_squad_subset()
raw = json.loads(SQUAD_SUBSET_PATH.read_text(encoding="utf-8"))
assert raw["license"] == "CC-BY-SA-4.0" and raw["modificationNotice"]
documents = pd.DataFrame(
    [{"document_id": item["document_id"], "title": item["title"], "context": item["context"]} for item in raw["documents"]]
)
questions = pd.DataFrame(
    [
        {
            "document_id": item["document_id"],
            "question": qa["question"],
            "answers": [answer["text"] for answer in qa["answers"]],
            "is_impossible": qa["is_impossible"],
            "source_qa_id": qa["source_qa_id"],
        }
        for item in raw["documents"]
        for qa in item["qas"]
    ]
)
validation_mask = questions.groupby("is_impossible", sort=False).cumcount().mod(2).eq(0)
validation_questions = questions[validation_mask].reset_index(drop=True)
test_questions = questions[~validation_mask].reset_index(drop=True)
assert len(documents) == 100 and len(questions) == 150
assert set(validation_questions["source_qa_id"]).isdisjoint(test_questions["source_qa_id"])
assert validation_questions["is_impossible"].value_counts().to_dict() == {False: 50, True: 25}
assert test_questions["is_impossible"].value_counts().to_dict() == {False: 50, True: 25}


## 1. Поиск документов

Обучите TF-IDF только на корпусе документов. Для каждого вопроса верните ранжированный список `document_id` и оценку сходства. Recall@k считается по наличию исходного документа SQuAD в первых k результатах. MRR (mean reciprocal rank) усредняет обратную позицию исходного документа: 1 для первого места, 1/2 для второго и 0, если документ не найден в пределах оценочного списка. Это упрощённая измеримая проверка для лабораторной, а не универсальная оценка поиска в продакшне.

In [ ]:
def build_retriever(corpus: pd.DataFrame):
    # TODO: верните обученный TfidfVectorizer и матрицу документов.
    raise NotImplementedError


def retrieve(question: str, *, top_k: int = 10) -> list[dict]:
    # TODO: верните список dict(document_id, score, context), score по убыванию.
    raise NotImplementedError


vectorizer, document_matrix = build_retriever(documents)


## 2. Простой ответ и отказ

Используйте готовую прозрачную функцию лексического ответа: она получает вопрос и первый найденный контекст, выбирает короткий фрагмент текста и возвращает оценку уверенности (`confidence`). Улучшать эту функцию не нужно — центральная работа лабораторной посвящена поиску и классификации ошибок. Для вопроса без ответа система должна уметь вернуть пустую строку. Порог отказа подбирается по сетке от 0 до 1 только на проверочной выборке, затем замораживается. Критерий выбора — среднее между долей успешных ответов на вопросах с ответом и долей корректных отказов на вопросах без ответа; при равенстве выбирается меньший порог. Порог успешного ответа `ANSWER_SUCCESS_F1 = 0.5` задаётся заранее и по тестовой выборке не меняется.

Не выдавайте это решение за генеративную модель: его задача — сделать причины ошибок видимыми. Классифицируйте каждый пример ровно в одну категорию:

- `retrieval_miss`: исходного документа нет в top-k;
- `bad_context`: исходный документ есть в top-k, но top-1 контекст другой;
- `unsupported_claim`: для вопроса без ответа система вернула непустой текст;
- `answering_failure`: top-1 контекст правильный, но token-F1 ответа ниже порога;
- `correct_abstention`: система корректно отказалась отвечать на вопрос без ответа;
- `success`: ответ на answerable-вопрос достиг порога token-F1.

Для вопросов с полем `is_impossible=true` сначала оценивается корректность отказа, поэтому они попадают в `unsupported_claim` или `correct_abstention` и не смешиваются с ошибками на вопросах, у которых есть ответ.

In [ ]:
ANSWER_SUCCESS_F1 = 0.5
EVALUATION_TOP_K = 10
THRESHOLD_GRID = np.linspace(0.0, 1.0, 21)


def extract_answer(question: str, context: str) -> tuple[str, float]:
    '''Прозрачная опорная функция: выбирает предложение с наибольшим пересечением слов.

    Это намеренно слабая функция ответа. Её не нужно улучшать: лабораторная проверяет
    контур оценки и локализацию ошибки, а не качество QA-модели.
    '''
    question_tokens = set(normalize_answer(question).split())
    candidates = [item.strip() for item in re.split(r"(?<=[.!?])\s+", context) if item.strip()]
    if not candidates or not question_tokens:
        return "", 0.0

    def overlap_score(candidate: str) -> float:
        candidate_tokens = set(normalize_answer(candidate).split())
        return len(question_tokens & candidate_tokens) / max(len(question_tokens), 1)

    best = max(candidates, key=overlap_score)
    confidence = float(np.clip(overlap_score(best), 0.0, 1.0))
    return best[:320], confidence


def evaluate_questions(frame: pd.DataFrame, *, abstain_threshold: float, top_k: int = EVALUATION_TOP_K) -> pd.DataFrame:
    # TODO: для каждого вопроса сохраните retrieved_ids, gold_rank, reciprocal_rank,
    # retrieval_hit_at_1/3, prediction, answer_f1, correct_abstention и один outcome
    # из шести bucket, описанных выше.
    raise NotImplementedError


# TODO: заполните таблицу для всей THRESHOLD_GRID. decision_score — среднее
# answer_success_rate и correct_abstention_rate; при равенстве выберите меньший threshold.
threshold_results = pd.DataFrame(
    columns=["threshold", "answer_success_rate", "correct_abstention_rate", "decision_score"]
)
ABSTAIN_THRESHOLD = None
validation_report = None


## 3. Замороженная проверка

После выбора порога один раз примените тот же контур к тестовой выборке. Отдельно покажите Recall@1/3, MRR по полному оценочному top-k, средний token-F1 на вопросах с ответом, долю правильных отказов и распределение всех фактически встретившихся исходов. Выведите до пяти примеров из каждой непустой категории ошибок. Не добавляйте искусственные строки ради заполнения категории.

In [ ]:
# TODO: ABSTAIN_THRESHOLD больше не меняется после этого места.
test_report = None
summary = None
failure_examples = None


In [ ]:
# Self-check: запускайте после выполнения всех TODO.
assert vectorizer is not None and document_matrix.shape[0] == len(documents)
probe = retrieve(validation_questions.iloc[0]["question"], top_k=EVALUATION_TOP_K)
assert len(probe) == EVALUATION_TOP_K and all({"document_id", "score", "context"} <= set(item) for item in probe)
assert [item["score"] for item in probe] == sorted([item["score"] for item in probe], reverse=True)
assert ABSTAIN_THRESHOLD is not None and 0 <= ABSTAIN_THRESHOLD <= 1
assert len(threshold_results) == len(THRESHOLD_GRID)
assert set(threshold_results.columns) == {
    "threshold", "answer_success_rate", "correct_abstention_rate", "decision_score",
}
assert np.allclose(sorted(threshold_results["threshold"]), THRESHOLD_GRID)
assert threshold_results[["answer_success_rate", "correct_abstention_rate", "decision_score"]].apply(
    lambda column: column.between(0, 1).all()
).all()
assert np.allclose(
    threshold_results["decision_score"],
    (threshold_results["answer_success_rate"] + threshold_results["correct_abstention_rate"]) / 2,
)
expected_threshold = threshold_results.sort_values(
    ["decision_score", "threshold"], ascending=[False, True]
).iloc[0]["threshold"]
assert np.isclose(ABSTAIN_THRESHOLD, expected_threshold)
required = {
    "source_qa_id", "retrieved_ids", "gold_rank", "reciprocal_rank",
    "retrieval_hit_at_1", "retrieval_hit_at_3", "prediction",
    "answer_f1", "correct_abstention", "outcome",
}
assert validation_report is not None and required <= set(validation_report.columns)
assert test_report is not None and required <= set(test_report.columns)
allowed_outcomes = {
    "retrieval_miss", "bad_context", "unsupported_claim",
    "answering_failure", "correct_abstention", "success",
}
assert set(test_report["outcome"]) <= allowed_outcomes
assert test_report["reciprocal_rank"].between(0, 1).all()
assert (test_report["retrieval_hit_at_1"] <= test_report["retrieval_hit_at_3"]).all()
assert test_report["answer_f1"].between(0, 1).all()
assert summary is not None and {
    "retrieval_recall_at_1", "retrieval_recall_at_3", "retrieval_mrr",
    "answer_f1", "abstention_accuracy", "outcome_counts",
} <= set(summary)
assert 0 <= summary["retrieval_mrr"] <= 1
assert failure_examples is not None
failure_buckets = {"retrieval_miss", "bad_context", "unsupported_claim", "answering_failure"}
assert set(failure_examples["outcome"]) <= failure_buckets
assert failure_examples.groupby("outcome").size().le(5).all()
for outcome in set(test_report["outcome"]) & failure_buckets:
    assert (failure_examples["outcome"] == outcome).any()
print(summary)
